# Can we predict a warranty claim within a year?
Follows `protocol.yaml`: fixed holdout (`holdout.csv`), stratified 5-fold CV on training rows only, choice by mean CV ROC AUC, holdout scored once.

In [ ]:
import json, hashlib
import numpy as np, pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from sklearn.base import clone

def sha(p): return hashlib.sha256(open(p,'rb').read()).hexdigest()
assert sha('data.csv')=='25552b73fd810f7dfa81a14849736291d84f91da2f8b5a58bcfaf0cc018962f4'
assert sha('holdout.csv')=='083821b17b5c7fdf81de46633ca08a7d601bd8266b269faa3911938d26bf66bf'
df=pd.read_csv('data.csv'); print(df.shape, df.claim_within_year.mean())

## Split
`holdout.csv` lists 0-based row positions in `data.csv`. `device_serial` is excluded (identifier). The other columns (channel, device age at sale, price, daily hours reported at registration) are recorded at sale/registration, so usable at prediction time.

In [ ]:
hold_idx=pd.read_csv('holdout.csv')['row'].to_numpy()
assert len(hold_idx)==1400 and len(set(hold_idx))==1400
train_idx=np.setdiff1d(np.arange(len(df)),hold_idx)
FEATURES=['channel','device_age_years','price_eur','daily_hours']
TARGET='claim_within_year'
X_tr,y_tr=df.loc[train_idx,FEATURES],df.loc[train_idx,TARGET]
X_ho,y_ho=df.loc[hold_idx,FEATURES],df.loc[hold_idx,TARGET]
print(len(train_idx),len(hold_idx), y_tr.mean(), y_ho.mean())

## Candidates (named before any scoring)
Preprocessing lives inside each pipeline, so it is fit within CV folds.

In [ ]:
cat=['channel']; num=['device_age_years','price_eur','daily_hours']
def prep(scale):
    return ColumnTransformer([('c',OneHotEncoder(handle_unknown='ignore'),cat),
                              ('n',StandardScaler() if scale else 'passthrough',num)])
candidates={
 'baseline_prior': Pipeline([('p',prep(False)),('m',DummyClassifier(strategy='prior'))]),
 'logistic_regression': Pipeline([('p',prep(True)),('m',LogisticRegression(max_iter=1000))]),
 'random_forest': Pipeline([('p',prep(False)),('m',RandomForestClassifier(n_estimators=300,min_samples_leaf=20,random_state=0,n_jobs=-1))]),
 'hist_gradient_boosting': Pipeline([('p',prep(False)),('m',HistGradientBoostingClassifier(max_depth=3,learning_rate=0.05,max_iter=150,random_state=0))]),
}
list(candidates)

## Cross-validation on training rows

In [ ]:
skf=StratifiedKFold(n_splits=5,shuffle=True,random_state=0)
folds=[(tr.tolist(),va.tolist()) for tr,va in skf.split(X_tr,y_tr)]
cv_scores={}
for name,est in candidates.items():
    s=[]
    for tr,va in folds:
        m=clone(est).fit(X_tr.iloc[tr],y_tr.iloc[tr])
        s.append(roc_auc_score(y_tr.iloc[va],m.predict_proba(X_tr.iloc[va])[:,1]))
    cv_scores[name]=s
cv_table=pd.DataFrame({k:{'mean':np.mean(v),'std':np.std(v)} for k,v in cv_scores.items()}).T
cv_table

## Choose by mean CV ROC AUC, refit on all training rows, score holdout once

In [ ]:
chosen=max(cv_scores,key=lambda k:np.mean(cv_scores[k]))
final=clone(candidates[chosen]).fit(X_tr,y_tr)
holdout_auc=roc_auc_score(y_ho,final.predict_proba(X_ho)[:,1])
print('chosen:',chosen,'CV AUC %.4f'%np.mean(cv_scores[chosen]),'| holdout AUC %.4f'%holdout_auc)

In [ ]:
# Row labels are 0-based positions in data.csv
evidence=dict(
 features=FEATURES,
 candidates=list(candidates),
 cv_folds=[[[int(train_idx[i]) for i in tr],[int(train_idx[i]) for i in va]] for tr,va in folds],
 cv_scores={k:[float(x) for x in v] for k,v in cv_scores.items()},
 chosen=chosen,
 holdout_rows=[int(i) for i in hold_idx],
 holdout_scores={chosen: float(holdout_auc)},
)
json.dump(evidence,open('evidence.json','w'))

## Conclusion
The headline is the holdout ROC AUC of the chosen candidate (printed above); compare it with the baseline (0.5) and the CV spread to judge whether the signal is real.